# Hamsokhan (هم‌سخن) — Advanced Data Preprocessing & Feature Engineering Laboratory
> **Target Environment:** Kaggle Notebook (GPU T4 Accelerator recommended)  
> **Objective:** Transform raw, noisy, heterogeneous Persian speech datasets into high-fidelity, mathematically aligned, zero-copy training tensors for real-time Speech-to-Speech modeling.  

---

## 🧪 The 6 Pillars of Hamsokhan Audio Feature Engineering:
1. **Persian Text Normalization:** Arabic glyph unification (`ي/ك -> ی/ک`), spoken expansion of numerals (`num2fawords`), stripping diacritics & invisible characters.
2. **Loudness Normalization (ITU-R BS.1770):** Standardizing signal power to strictly **-20 dBFS** to stabilize gradient updates and eliminate dynamic range mismatches.
3. **Smart VAD Trimming with Room Tone:** Preserving **160ms onset silence** (prevents clipping Persian plosives `/پ/`, `/ت/`, `/ک/`) and **240/320ms offset** (preserves sentence-final breath/devoicing) with a **15ms Hann cosine taper**.
4. **160ms Macro-Packet Alignment:** Quantizing audio duration to exact multiples of 160ms ($3840$ samples @ 24kHz = $15$ Vocos Mel frames = $8$ Whisper frames = $2$ Qwen SLM tokens) with zero fractional lookahead.
5. **Continuous Pitch & Prosody Engineering (log-F0):** Extracting continuous fundamental frequency ($F_0$) contours via YIN with zero-mean unit-variance normalization on voiced frames.
6. **Turn-Taking Soft Ramp Labels (PersianVox):** Generating smooth ramp targets `[0.15, 0.35, 0.65, 1.0, 1.0, 1.0]` for End-of-Turn and tagging thinking pauses ($< 720$ms) as `0.0` (Hard Negatives).

In [ ]:
# ==========================================================================
# STEP 1: Environment Setup & Clone Repository
# ==========================================================================
!git clone -b mahdi https://github.com/Abolfazl-Afkhami/Hamsokhan.git
%cd Hamsokhan
!pip install -q transformers datasets torchaudio vocos soundfile librosa safetensors matplotlib num2fawords

In [ ]:
# ==========================================================================
# STEP 2: Interactive Feature Engineering Sanity Check & Visualization
# ==========================================================================
import math
import numpy as np
import torch
import torchaudio
import matplotlib.pyplot as plt
import IPython.display as ipd

from src.data.mel_extractor import VocosAlignedMelExtractor, smart_persian_vad_trim
from src.nlp.normalizer import PersianTextNormalizer
from scripts.kaggle_dataset_builder import KaggleDatasetBuilder

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Initialized on device: {device}")

# 1. Test Text Normalization
normalizer = PersianTextNormalizer(convert_numbers=True)
raw_persian = "در سال 1405، سیستم هوش مصنوعی هم‌سخن با 24kHz تست شد."
cleaned_persian = normalizer.normalize(raw_persian)
print(f"\n[Text Normalization Test]")
print(f"Raw:     {raw_persian}")
print(f"Cleaned: {cleaned_persian}")

# 2. Synthesize Benchmark Audio with Pitch Glide (140Hz -> 220Hz)
sr = 24000
dur = 3.2  # 3.2s = exactly 20 macro-packets of 160ms
t = torch.linspace(0, dur, int(dur * sr))
f0_glide = 140.0 + 80.0 * (t / dur)
raw_wav = (0.5 * torch.sin(2 * math.pi * f0_glide * t) + 0.25 * torch.sin(4 * math.pi * f0_glide * t)).unsqueeze(0)

# 3. Loudness Normalization to -20 dBFS
norm_wav = KaggleDatasetBuilder.normalize_loudness_bs1770(raw_wav, target_dbfs=-20.0)

# 4. Vocos-Aligned 100-Channel Mel Extraction
mel_extractor = VocosAlignedMelExtractor(sample_rate=24000, n_mels=100).to(device)
mel = mel_extractor(norm_wav.to(device)).squeeze(0).cpu()  # [100, T_mel]

# 5. Continuous log-F0 Pitch Contour Extraction
f0_contour = KaggleDatasetBuilder.extract_log_f0(norm_wav, sr=24000, hop_length=256, target_len=mel.shape[-1])

print(f"\n[Acoustic Feature Dimensions]")
print(f"✓ Normalized Waveform Shape: {norm_wav.shape} ({norm_wav.shape[-1]/sr:.2f}s)")
print(f"✓ Vocos Mel-Spectrogram Shape: {mel.shape} (100 bins, {mel.shape[-1]} frames @ 93.75Hz)")
print(f"✓ Pitch Contour (log-F0) Shape: {f0_contour.shape}")
assert mel.shape[-1] % 15 == 0, "Verification failed: Mel frames must be multiple of 15!"

In [ ]:
# ==========================================================================
# STEP 3: Multi-Modal Feature Inspection Plot
# ==========================================================================
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=False)

# Plot 1: Waveform (Loudness Normalized @ -20 dBFS)
axes[0].plot(np.linspace(0, dur, norm_wav.shape[-1]), norm_wav.squeeze(0).numpy(), color="#1f77b4", alpha=0.8)
axes[0].set_title("1. Cleaned & Loudness Normalized Waveform (ITU-R BS.1770 -20 dBFS)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Amplitude")
axes[0].grid(True, alpha=0.3)

# Plot 2: Vocos 100-Channel Mel-Spectrogram
im = axes[1].imshow(mel.float().numpy(), aspect="auto", origin="lower", cmap="magma")
axes[1].set_title("2. Vocos-Aligned 100-Channel Log Mel-Spectrogram (160ms Macro-Packet Aligned)", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Mel Frequency Bins (0..99)")
plt.colorbar(im, ax=axes[1], format="%+.1f dB")

# Plot 3: Normalized Pitch Contour (log-F0)
axes[2].plot(f0_contour.float().numpy(), color="#2ca02c", linewidth=2, label="Extracted log-F0")
axes[2].set_title("3. Continuous Normalized Pitch Contour (F0 Melody Preservation)", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Time Frames (at 93.75Hz)")
axes[2].set_ylabel("Normalized F0")
axes[2].legend(loc="upper left")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("🎧 Listen to the benchmark audio sample:")
ipd.display(ipd.Audio(norm_wav.numpy(), rate=24000))

In [ ]:
# ==========================================================================
# STEP 4: Execute Autonomous Preprocessing Pipeline (Thomcles 24kHz Synthesis)
# ==========================================================================
# Supports partition splitting via --num_parts and --part_idx to keep disk usage under 15GB per run
!python scripts/kaggle_dataset_builder.py \
    --dataset_type thomcles \
    --output_dir /kaggle/working/hamsokhan_shards \
    --part_idx 0 \
    --num_parts 1 \
    --max_hours 10.0 \
    --shard_size 500

In [ ]:
# ==========================================================================
# STEP 5: Execute Autonomous Preprocessing Pipeline (PersianVox Turn-Taking)
# ==========================================================================
# Extracts conversational turn boundaries & soft ramp labels with hard negative pause tagging
!python scripts/kaggle_dataset_builder.py \
    --dataset_type persianvox \
    --output_dir /kaggle/working/hamsokhan_shards \
    --part_idx 0 \
    --num_parts 1 \
    --max_hours 5.0 \
    --shard_size 500

In [ ]:
# ==========================================================================
# STEP 6: Quality Audit & Kaggle Dataset Publishing Guide
# ==========================================================================
from pathlib import Path
from safetensors.torch import safe_open

out_dir = Path("/kaggle/working/hamsokhan_shards")
shards = sorted(list(out_dir.glob("*.safetensors")))
total_bytes = sum(f.stat().st_size for f in shards)
total_gb = total_bytes / (1024 ** 3)

print("=" * 60)
print(f"📊 PREPROCESSING AUDIT REPORT")
print("=" * 60)
print(f"✓ Total Shards Generated:    {len(shards)}")
print(f"✓ Total Storage Footprint:   {total_gb:.2f} GB (Max quota: 20 GB)")

if shards:
    first_shard = shards[0]
    with safe_open(first_shard, framework="pt", device="cpu") as f:
        w_flat = f.get_slice("whisper_flat").get_shape()
        m_flat = f.get_slice("mel_flat").get_shape()
        f0_flat = f.get_slice("f0_flat").get_shape()
        lbl_flat = f.get_slice("labels_flat").get_shape()
        idx_tbl = f.get_tensor("index_table")
        print(f"\n🔍 Shard Sample Schema ({first_shard.name}):")
        print(f"  - whisper_flat: {w_flat} (FP16 @ 50Hz)")
        print(f"  - mel_flat:     {m_flat} (100 bins FP16 @ 93.75Hz)")
        print(f"  - f0_flat:      {f0_flat} (Normalized Pitch Contour)")
        print(f"  - labels_flat:  {lbl_flat} (Soft Ramp Labels)")
        print(f"  - index_table:  {idx_tbl.shape} (Zero-Copy Utterance Boundaries)")

print("\n=======================================================")
print("🚀 HOW TO PUBLISH THIS AS A KAGGLE DATASET:")
print("1. Click 'Save Version' at the top right of this notebook.")
print("2. Select 'Save & Run All (Commit)'.")
print("3. Once finished, go to the notebook Output panel.")
print("4. Click 'Create Dataset' -> Name it 'hamsokhan-curated-shards'.")
print("5. In your training notebook, add it via '+ Add Input'!")
print("=======================================================")